# Lab 1: The Machine Learning Workflow
**Machine Learning (MSc), AAU AAiT SiTE** · Lecture chapter 1 · Lab instructor: Abel T. · Course instructor: Dr Fantahun B.

In this lab you take one dataset through a complete machine learning workflow:

| Part | Topic | Approx. time |
|---|---|---|
| A | Explore the data | 25 min |
| B | Preprocess inside a `Pipeline`, without data leakage | 35 min |
| C | Linear regression: normal equation and gradient descent | 50 min |
| D | Under- and overfitting with polynomial models | 35 min |
| E | k-fold cross-validation and a final test score | 35 min |

**How to work.** Complete every cell marked `# TODO` (replace `None` or `raise NotImplementedError`). Each part ends with a **check cell** of `assert` statements; when it prints ✅ you may move on. Answer each **Question** in the markdown cell below it.

**Submission.** The completed `.ipynb` and a PDF export, within one week. The notebook must run from top to bottom without errors (*Kernel → Restart & Run All*).

In [ ]:
# Setup: run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.precision", 3)
print("Setup complete")

### The dataset
We use scikit-learn's **Diabetes** dataset in raw (unscaled) form: 442 patients, 10 baseline measurements and a target that measures disease progression one year later.

| Column | Meaning |
|---|---|
| `age` | age in years |
| `sex` | recorded as two codes; we treat it as a **categorical** feature |
| `bmi` | body mass index |
| `bp` | mean blood pressure |
| `s1` … `s6` | six blood serum measurements |
| `target` | disease progression after one year (the value we predict) |

Real data is rarely clean, so the next cell removes about 5% of the values in `bmi` and `bp` at random (fixed seed, so everyone has the same data). Do not change this cell.

In [ ]:
raw = load_diabetes(as_frame=True, scaled=False)
df = raw.frame.copy()
df["sex"] = df["sex"].map({1.0: "group_1", 2.0: "group_2"})   # categorical feature

rng = np.random.default_rng(SEED)
for col in ["bmi", "bp"]:
    idx = rng.choice(len(df), size=int(0.05 * len(df)), replace=False)
    df.loc[idx, col] = np.nan

df.head()

---
## Part A: Explore the data
Before fitting anything, find out what the data looks like: its size, types, missing values, ranges, distributions and the relationships between features and the target.

**A1.** Compute the number of rows and columns, and a `Series` with the number of missing values per column. Then display summary statistics with `describe()`.

In [ ]:
# TODO: fill in n_rows, n_cols and missing_counts
n_rows, n_cols = None, None
missing_counts = None      # hint: df.isna()

print(f"{n_rows} rows, {n_cols} columns")
print("\nMissing values per column:")
print(missing_counts[missing_counts > 0])
print("\nColumn types:")
print(df.dtypes.value_counts())
df.describe()

**A2.** Plot a histogram of every numeric column (including the target) and a bar chart of the categorical column `sex`.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.drop("target").tolist()
categorical_cols = ["sex"]

# TODO: histograms of numeric_cols + ["target"] (hint: df[...].hist(bins=20, figsize=(12, 8)))
raise NotImplementedError
plt.suptitle("Distributions of numeric columns")
plt.tight_layout()
plt.show()

df["sex"].value_counts().plot.bar(title="sex", rot=0)
plt.show()

**A3.** Compute the correlation matrix of the numeric columns and the target. Find the feature with the **largest absolute correlation** with the target.

In [ ]:
# TODO: corr = correlation matrix of numeric_cols + ["target"]
#       top_feature = name of the feature with the largest |correlation| with the target
corr = None
top_feature = None

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation matrix")
ax.grid(False)
plt.show()

print("Correlation with target:\n", corr["target"].drop("target").sort_values(ascending=False).round(3))
print("\nMost correlated feature:", top_feature)

plt.scatter(df[top_feature], df["target"], s=12, alpha=0.6)
plt.xlabel(top_feature); plt.ylabel("target"); plt.title(f"target vs {top_feature}")
plt.show()

In [ ]:
# Check Part A
assert (n_rows, n_cols) == (442, 11), "shape is wrong"
assert isinstance(missing_counts, pd.Series), "missing_counts should be a pandas Series"
assert missing_counts["bmi"] == 22 and missing_counts["bp"] == 22, "missing counts are wrong"
assert corr.shape == (10, 10), "corr should cover the 10 numeric columns: 9 features plus the target"
assert top_feature == "bmi", "check how you rank the correlations (absolute value?)"
print("✅ Part A checks passed")

> **Question A.** (a) Which two features are most strongly related to the target, and is the relationship positive or negative? (b) Look at the ranges in `describe()`: why would these different scales be a problem for gradient descent or for distance-based methods? (c) Which feature is negatively correlated with the target?

**Your answer:**

*Write your answer here.*

---
## Part B: Preprocess without data leakage
**Data leakage** happens when information from the validation or test data influences training, for example when a scaler's mean and standard deviation, or an imputer's median, are computed on the full dataset. The scores you report then look better than what the model will achieve on genuinely new data.

The rule: **split first, then fit every preprocessing step on the training set only**. A scikit-learn `Pipeline` enforces this automatically.

**B1.** Split the data into 60% training, 20% validation and 20% test. Use two calls to `train_test_split` with `random_state=SEED`: first hold out 20% as test, then take 25% of the remainder as validation (0.25 × 0.8 = 0.2).

In [ ]:
X = df.drop(columns="target")
y = df["target"]

# TODO: create X_train, X_val, X_test, y_train, y_val, y_test
X_dev, X_test, y_dev, y_test = None, None, None, None   # dev = train + val
X_train, X_val, y_train, y_val = None, None, None, None

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)

**B2.** Build a `ColumnTransformer` called `preprocessor`:
* numeric columns: `SimpleImputer(strategy="median")` then `StandardScaler()`
* categorical columns: `SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(drop="if_binary", handle_unknown="ignore")`

Fit it on `X_train` **only**, then transform the three sets into NumPy arrays `Xtr`, `Xva`, `Xte`.

*Why `drop="if_binary"`?* A binary feature one-hot encoded into two columns that always sum to 1 duplicates the intercept, which makes $X^\top X$ singular in Part C.

In [ ]:
# TODO: build preprocessor, fit on the training data, transform all three sets
numeric_pipe = None
categorical_pipe = None
preprocessor = None
# fit the preprocessor here (training data only!)

Xtr = preprocessor.transform(X_train)
Xva = preprocessor.transform(X_val)
Xte = preprocessor.transform(X_test)
ytr, yva, yte = y_train.to_numpy(), y_val.to_numpy(), y_test.to_numpy()

feature_names = preprocessor.get_feature_names_out()
print("Features after preprocessing:", list(feature_names))
print("Xtr:", Xtr.shape)

**B3. See the leak.** The next cell compares the median of `bmi` learned from the training set with the one you would learn from the whole dataset. Run it and note that the two differ: fitting on everything quietly lets test rows shape your preprocessing.

In [ ]:
median_train = preprocessor.named_transformers_["num"].named_steps["impute"].statistics_[numeric_cols.index("bmi")]
median_all = X["bmi"].median()
print(f"bmi median learned from training data: {median_train:.2f}")
print(f"bmi median learned from ALL data     : {median_all:.2f}   <- uses test rows: leakage")

print("\nMean of scaled numeric features (train):", Xtr[:, :len(numeric_cols)].mean(axis=0).round(3))
print("Mean of scaled numeric features (val)  :", Xva[:, :len(numeric_cols)].mean(axis=0).round(3))

In [ ]:
# Check Part B
assert len(X_train) == 264 and len(X_val) == 89 and len(X_test) == 89, "split sizes are wrong"
assert Xtr.shape == (264, 10), "expected 9 scaled numeric columns + 1 one-hot column"
assert not np.isnan(Xtr).any() and not np.isnan(Xte).any(), "missing values remain"
assert np.allclose(Xtr[:, :9].mean(axis=0), 0, atol=1e-8), "numeric features should have mean 0 on the TRAINING set"
assert np.allclose(Xtr[:, :9].std(axis=0), 1, atol=1e-8), "numeric features should have std 1 on the TRAINING set"
print("✅ Part B checks passed")

> **Question B.** (a) Why are the validation means in B3 not exactly zero, and why is that correct? (b) Give one concrete example of leakage other than scaling or imputation.

**Your answer:**

*Write your answer here.*

---
## Part C: Linear regression from scratch
The model is $\hat{y} = \mathbf{X}\mathbf{w}$, where $\mathbf{X}$ has a leading column of ones so that $w_0$ is the intercept. We minimise the mean squared error

$$J(\mathbf{w}) = \frac{1}{n}\lVert \mathbf{X}\mathbf{w} - \mathbf{y} \rVert^2 .$$

**Normal equation.** Setting the gradient to zero gives $\mathbf{X}^\top\mathbf{X}\,\mathbf{w} = \mathbf{X}^\top\mathbf{y}$. Solve this system with `np.linalg.solve` (more stable than forming the inverse).

**Gradient descent.** Start from $\mathbf{w} = \mathbf{0}$ and repeat
$$\mathbf{w} \leftarrow \mathbf{w} - \eta\,\nabla J(\mathbf{w}), \qquad \nabla J(\mathbf{w}) = \frac{2}{n}\mathbf{X}^\top(\mathbf{X}\mathbf{w} - \mathbf{y}).$$

**C1.** Complete the helper functions and the normal equation.

In [ ]:
def add_bias(X):
    # Prepend a column of ones to X
    # TODO
    raise NotImplementedError

def predict(X, w):
    # X already contains the bias column
    return X @ w

def mse(y_true, y_pred):
    # TODO: mean squared error
    raise NotImplementedError

def fit_normal_equation(X, y):
    # Return w that solves (X^T X) w = X^T y. X already contains the bias column
    # TODO
    raise NotImplementedError

Xtr_b, Xva_b, Xte_b = add_bias(Xtr), add_bias(Xva), add_bias(Xte)
w_ne = fit_normal_equation(Xtr_b, ytr)
print("Normal equation weights:", w_ne.round(2))
print(f"Train RMSE: {np.sqrt(mse(ytr, predict(Xtr_b, w_ne))):.2f}   Val RMSE: {np.sqrt(mse(yva, predict(Xva_b, w_ne))):.2f}")

**C2.** Implement batch gradient descent. Record the training MSE at every iteration in `history`. We run 10,000 iterations: the loss drops within a few dozen, but the weights of the strongly correlated features `s1` and `s2` take much longer to settle. Watch the convergence plot.

In [ ]:
def fit_gradient_descent(X, y, lr=0.1, n_iters=10000):
    # Returns (w, history). X already contains the bias column
    n, d = X.shape
    w = np.zeros(d)
    history = []
    for _ in range(n_iters):
        # TODO: compute the gradient, update w, append the current MSE to history
        raise NotImplementedError
    return w, np.array(history)

w_gd, hist = fit_gradient_descent(Xtr_b, ytr, lr=0.1, n_iters=10000)
print("Gradient descent weights:", w_gd.round(2))
print("Max |w_gd - w_ne|:", np.abs(w_gd - w_ne).max().round(4))

plt.plot(np.arange(1, len(hist) + 1), hist)
plt.xscale("log")
plt.axhline(mse(ytr, predict(Xtr_b, w_ne)), color="k", ls="--", label="normal equation optimum")
plt.yscale("log"); plt.xlabel("iteration (log scale)"); plt.ylabel("training MSE (log scale)")
plt.title("Gradient descent convergence"); plt.legend(); plt.show()

**C3.** Study the learning rate. Run gradient descent for 300 iterations with each learning rate below and plot the loss curves on one figure. Which rate is too small, which converges well, and which diverges?

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5]
plt.figure()
for lr in learning_rates:
    _, h = fit_gradient_descent(Xtr_b, ytr, lr=lr, n_iters=300)
    plt.plot(h, label=f"lr = {lr}")
plt.yscale("log"); plt.ylim(1e3, 1e6)
plt.xlabel("iteration"); plt.ylabel("training MSE (log scale)")
plt.title("Effect of the learning rate"); plt.legend(); plt.show()

**C4.** Fit scikit-learn's `LinearRegression` on `Xtr` (it adds its own intercept) and compare it with your two solutions.

In [ ]:
# TODO: fit LinearRegression on Xtr, ytr and build w_sk = [intercept, coef_1, ..., coef_d]
lr_sk = None
w_sk = None

comparison = pd.DataFrame({"normal eq.": w_ne, "grad. descent": w_gd, "scikit-learn": w_sk},
                          index=["intercept"] + list(feature_names))
display(comparison.round(3))

In [ ]:
# Check Part C
assert add_bias(np.zeros((3, 2))).shape == (3, 3) and np.all(add_bias(np.zeros((3, 2)))[:, 0] == 1)
assert np.isclose(mse(np.array([1., 2.]), np.array([1., 4.])), 2.0)
assert np.allclose(w_ne, w_sk, atol=1e-6), "normal equation does not match scikit-learn"
assert np.allclose(w_gd, w_sk, atol=1e-3), "gradient descent has not converged to the scikit-learn solution"
assert np.all(np.diff(hist) <= 1e-9), "with lr = 0.1 the training loss should never increase"
print("✅ Part C checks passed")

> **Question C.** (a) Both methods give the same weights. When would you prefer gradient descent over the normal equation? (b) Describe what happened with each learning rate in C3. (c) Why do we need standardised features for gradient descent, and why does it still need thousands of iterations here? (d) Which features have the largest weights, and can you compare weights directly here? Why?

**Your answer:**

*Write your answer here.*

---
## Part D: Under- and overfitting
To see under- and overfitting clearly we use a small synthetic problem where we know the truth: $y = \sin(\pi x) + \varepsilon$ for $x \in [-1, 1]$, with $\varepsilon \sim \mathcal{N}(0, 0.3^2)$. We have only **20 training points** and 200 validation points.

**D1.** For each polynomial degree from 0 to 14, fit `Pipeline([PolynomialFeatures(degree), LinearRegression()])` on the training data and record training and validation MSE.

In [ ]:
def true_fn(x):
    return np.sin(np.pi * x)

rng_d = np.random.default_rng(SEED)
x_tr = np.sort(rng_d.uniform(-1, 1, 20));  y_tr = true_fn(x_tr) + rng_d.normal(0, 0.3, 20)
x_va = rng_d.uniform(-1, 1, 200);         y_va = true_fn(x_va) + rng_d.normal(0, 0.3, 200)

degrees = list(range(0, 15))
train_err, val_err = [], []

def make_poly(degree):
    return Pipeline([("poly", PolynomialFeatures(degree)), ("lin", LinearRegression())])

for d in degrees:
    # TODO: fit make_poly(d) on x_tr, y_tr (reshape x to a column), append train and val MSE
    raise NotImplementedError

best_degree = degrees[int(np.argmin(val_err))]
print("Best degree on validation:", best_degree)

plt.plot(degrees, train_err, "o-", label="training MSE")
plt.plot(degrees, val_err, "s-", label="validation MSE")
plt.axhline(0.3 ** 2, color="gray", ls=":", label="noise level (0.09)")
plt.axvline(best_degree, color="k", ls="--", alpha=0.5)
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.ylabel("MSE (log scale)")
plt.title("Training and validation error against model complexity"); plt.legend(); plt.show()

**D2.** Visualise three fits: an underfitting model, the best model and an overfitting model.

In [ ]:
x_plot = np.linspace(-1, 1, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, d, label in zip(axes, [1, best_degree, 14], ["underfitting", "good fit", "overfitting"]):
    m = make_poly(d).fit(x_tr.reshape(-1, 1), y_tr)
    ax.plot(x_plot, true_fn(x_plot), "g--", label="true function")
    ax.plot(x_plot, m.predict(x_plot), "r-", label=f"degree {d}")
    ax.scatter(x_tr, y_tr, s=25, label="training points")
    ax.set_ylim(-2, 2); ax.set_title(f"Degree {d}: {label}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**D3. More data.** Repeat D1 with 200 training points instead of 20 (same noise) and plot both validation curves on one figure. What happens to the overfitting region?

In [ ]:
# TODO: generate 200 training points, recompute the validation MSE for every degree as val_err_big
x_tr_big, y_tr_big = None, None
val_err_big = []

plt.plot(degrees, val_err, "s-", label="validation MSE, n_train = 20")
plt.plot(degrees, val_err_big, "^-", label="validation MSE, n_train = 200")
plt.axhline(0.3 ** 2, color="gray", ls=":", label="noise level")
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.ylabel("MSE (log scale)")
plt.title("More training data reduces overfitting"); plt.legend(); plt.show()

In [ ]:
# Check Part D
assert len(train_err) == len(degrees) and len(val_err) == len(degrees)
assert train_err[-1] < train_err[1], "training error should fall as the degree grows"
assert 3 <= best_degree <= 7, f"unexpected best degree {best_degree}"
assert val_err[-1] > 2 * min(val_err), "degree 14 should clearly overfit with 20 points"
assert len(val_err_big) == len(degrees) and val_err_big[-1] < val_err[-1], "more data should reduce overfitting"
print("✅ Part D checks passed")

> **Question D.** (a) Mark the underfitting and overfitting regions on the error plot and explain each in terms of bias and variance. (b) Why does training error keep falling while validation error rises? (c) Why can't the validation MSE go much below 0.09? (d) What did ten times more data change?

**Your answer:**

*Write your answer here.*

---
## Part E: Compare models with k-fold cross-validation
One validation split gives one noisy number. **k-fold cross-validation** splits the development data (training + validation, 353 rows) into k folds, trains on k − 1 folds and scores on the held-out fold, k times. We report the **mean and standard deviation** of the k scores.

**E1.** Implement k-fold CV yourself. Use `KFold(n_splits=k, shuffle=True, random_state=SEED)` to generate the indices, `clone(model)` to get a fresh copy for each fold, and return the array of fold RMSEs.

In [ ]:
def my_cross_val_rmse(model, X, y, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, val_idx in kf.split(X):
        # TODO: fit a fresh clone on the training fold, append the RMSE on the validation fold
        raise NotImplementedError
    return np.array(scores)

def with_preprocessing(estimator, extra_steps=()):
    # Full pipeline: preprocessing is refitted inside every fold, so there is no leakage
    return Pipeline([("prep", clone(preprocessor)), *extra_steps, ("model", estimator)])

linear_all = with_preprocessing(LinearRegression())
my_scores = my_cross_val_rmse(linear_all, X_dev, y_dev, k=5)
sk_scores = -cross_val_score(linear_all, X_dev, y_dev, cv=KFold(5, shuffle=True, random_state=SEED),
                             scoring="neg_root_mean_squared_error")
print("my folds     :", my_scores.round(2))
print("scikit-learn :", sk_scores.round(2))

**E2.** Compare five models with 5-fold CV and collect the results in a DataFrame with columns `mean RMSE` and `std RMSE`, sorted by mean RMSE:

1. **Baseline**: always predict the training mean (`DummyRegressor`)
2. **Linear (bmi only)**
3. **Linear (all features)**
4. **Polynomial degree 2**
5. **Polynomial degree 3**

In [ ]:
bmi_only = Pipeline([
    ("select", ColumnTransformer([("bmi", Pipeline([("impute", SimpleImputer(strategy="median")),
                                                    ("scale", StandardScaler())]), ["bmi"])])),
    ("model", LinearRegression()),
])

models = {
    "Baseline (mean)": with_preprocessing(DummyRegressor(strategy="mean")),
    "Linear (bmi only)": bmi_only,
    "Linear (all features)": with_preprocessing(LinearRegression()),
    "Polynomial degree 2": with_preprocessing(LinearRegression(), [("poly", PolynomialFeatures(2, include_bias=False))]),
    "Polynomial degree 3": with_preprocessing(LinearRegression(), [("poly", PolynomialFeatures(3, include_bias=False))]),
}

# TODO: run my_cross_val_rmse for every model; store all fold scores in cv_scores (dict name -> array)
#       and build the results DataFrame
cv_scores = {}
results = None
display(results.round(2))

plt.figure(figsize=(9, 4))
names = list(cv_scores)
plt.boxplot([cv_scores[n] for n in names])
plt.xticks(range(1, len(names) + 1), names, rotation=15)
plt.yscale("log"); plt.ylabel("RMSE across 5 folds (log scale)"); plt.title("5-fold cross-validation")
plt.tight_layout(); plt.show()

**E3. The final test.** Choose the model with the lowest mean CV RMSE, fit it on the full development set, and evaluate it **once** on the test set.

In [ ]:
# TODO: best_name = name of the best model; fit it on X_dev, y_dev; compute test_rmse on X_test, y_test
best_name = None
final_model = None
test_rmse = None
print(f"Selected model : {best_name}")
print(f"CV RMSE        : {results.loc[best_name, 'mean RMSE']:.2f} ± {results.loc[best_name, 'std RMSE']:.2f}")
print(f"Test RMSE      : {test_rmse:.2f}")

In [ ]:
# Check Part E
assert np.allclose(my_scores, sk_scores), "your CV does not match cross_val_score"
assert set(cv_scores) == set(models) and all(len(s) == 5 for s in cv_scores.values())
assert list(results.columns) == ["mean RMSE", "std RMSE"]
assert results["mean RMSE"].is_monotonic_increasing, "sort the results by mean RMSE"
assert best_name == "Linear (all features)", "check which model has the lowest mean RMSE"
assert 45 < test_rmse < 65, "test RMSE looks wrong"
print("✅ Part E checks passed")

> **Question E.** (a) Report the best model as "mean ± std" and say whether its advantage over the second-best model is convincing given the spread. (b) Why does polynomial degree 3 do so badly here, when degree 3 or more was best in Part D? (c) Why must the preprocessing be inside the pipeline that is cross-validated? (d) Why do we use the test set only once, and what would go wrong if we used it to choose between models?

**Your answer:**

*Write your answer here.*

---
## Summary
* **Explore first**: scales, missing values and correlations shape every later decision.
* **Split first, then preprocess**, with all statistics learned from the training data inside a `Pipeline`.
* The **normal equation** and **gradient descent** reach the same solution; gradient descent needs scaled features and a sensible learning rate.
* **Training error alone always rewards complexity**; validation error reveals under- and overfitting.
* **Cross-validation** gives a mean and a spread; the **test set** is used once, at the very end.

**Next lab:** linear classification (perceptron, logistic regression, evaluation metrics).